In [0]:
stg_posts_df = spark.table('stackexchange_dataset.default.stg_posts')
raw_users_df = spark.table('stackexchange_dataset.default.raw_users')

In [0]:
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def posts_users_OBT(stg_posts_df: DataFrame, raw_users_df: DataFrame) -> DataFrame:
    return(
        stg_posts_df
            .alias('posts')
            .withColumnRenamed('CreationDate', 'PostCreationDate')
            .join(
                raw_users_df
                    .alias('users')
                    .withColumnRenamed('CreationDate', 'UserCreationDate'),
                on = F.col('posts.OwnerUserId') == F.col('users.Id'),
                how = 'left'
            )
    )

marts_posts_users_df = posts_users_OBT(stg_posts_df, raw_users_df) 

In [0]:
(
    marts_posts_users_df
        .write
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable('stackexchange_dataset.default.marts_posts_users')
)